In [ ]:
import numpy as np
import h5py
from scipy.signal import butter, filtfilt, iirnotch

# ─────────────────────────────────────────────────────────────
# COSTANTI
# ─────────────────────────────────────────────────────────────
FS = 125
N_SAMPLES = 250
N_LEADS = 12

# Se le label nel tuo H5 sono già intere 0..7:
NORMAL_LABEL = 0


FLATLINE_STD_THR = 0.05
NOISE_STD_THR = 5.0
AMPLITUDE_MAX = 10.0
AMPLITUDE_MIN = 0.05
BASELINE_MAX = 3.0
CLIPPING_RATIO = 0.05


def load_h5_data(file_path):
    with h5py.File(file_path, 'r') as f:
        x = np.array(f['X'])
        y = np.array(f['Y'])
    return x, y


def bandpass_filter(signal, fs=FS, lowcut=0.5, highcut=40.0, order=4):
    nyq = fs / 2.0
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, signal)


def notch_filter(signal, fs=FS, freq=50.0, Q=30.0):
    b, a = iirnotch(freq / (fs / 2.0), Q)
    return filtfilt(b, a, signal)


def check_lead_quality(lead_signal):
    checks = {}

    if np.any(~np.isfinite(lead_signal)):
        return {'valid': False, 'reason': 'nan_inf', 'checks': {'nan_inf': True}}

    std_val = float(np.std(lead_signal))
    checks['flatline'] = std_val < FLATLINE_STD_THR
    checks['excess_noise'] = std_val > NOISE_STD_THR

    pp_amplitude = float(np.max(lead_signal) - np.min(lead_signal))
    checks['low_amplitude'] = pp_amplitude < AMPLITUDE_MIN

    n_clipped = np.sum(np.abs(lead_signal) >= AMPLITUDE_MAX)
    checks['clipping'] = (n_clipped / len(lead_signal)) > CLIPPING_RATIO

    filtered = bandpass_filter(lead_signal)
    dc_offset = float(np.abs(np.mean(filtered)))
    checks['baseline_wander'] = dc_offset > BASELINE_MAX

    mean_val = np.mean(filtered)
    std_f = np.std(filtered) + 1e-8
    skewness = float(np.mean(((filtered - mean_val) / std_f) ** 3))
    checks['high_skewness'] = abs(skewness) > 5.0

    fail_reasons = [k for k, v in checks.items() if v]
    valid = len(fail_reasons) == 0
    reason = ', '.join(fail_reasons) if fail_reasons else 'OK'

    return {'valid': valid, 'reason': reason, 'checks': checks}


def check_ecg_quality(ecg, min_valid_leads=9):
    # Accetta sia (12,250) sia (250,12)
    if ecg.ndim != 2:
        raise ValueError(f"Shape non valida: {ecg.shape}")

    if ecg.shape == (N_SAMPLES, N_LEADS):
        ecg = ecg.T
    elif ecg.shape != (N_LEADS, N_SAMPLES):
        raise ValueError(f"Shape non valida: {ecg.shape}, atteso (12,250) o (250,12)")

    lead_results = []
    for i in range(N_LEADS):
        result = check_lead_quality(ecg[i])
        result['lead_idx'] = i
        lead_results.append(result)

    valid_leads = sum(r['valid'] for r in lead_results)
    invalid_indices = [r['lead_idx'] for r in lead_results if not r['valid']]
    global_valid = valid_leads >= min_valid_leads

    return {
        'global_valid': global_valid,
        'valid_leads': valid_leads,
        'total_leads': N_LEADS,
        'invalid_lead_indices': invalid_indices,
        'lead_results': lead_results
    }


# prendiamo solo quelli con etichetta normale (0 o 'normale')
def select_normal_records(x, y, normal_label=NORMAL_LABEL):
    # Gestione label bytes/stringa se necessario
    if y.dtype.kind in {'S', 'U', 'O'}:
        y_clean = np.array([
            label.decode('utf-8') if isinstance(label, bytes) else label
            for label in y
        ])
    else:
        y_clean = y

    mask = (y_clean == normal_label)
    return x[mask], y_clean[mask], mask




In [29]:
MAPPING_INV = {
    'normale': 0,
    'LA-RA': 1, 
    'RA-LL': 2, 
    'LA-LL': 3, 
    'ROT_ORARIA': 4, 
    'ROT_ANTIORARIA': 5,
    'RL-RA': 6, 
    'RL-LA': 7
}
MAPPING_BY_ID = {v: k for k, v in MAPPING_INV.items()}
# ─────────────────────────────────────────────────────────────
# ESEMPIO USO
# ─────────────────────────────────────────────────────────────
dataset= "val_dataset.h5"
x_all, y_all = load_h5_data(dataset)
print("analizzando", dataset)

# per prendere solo le label normali (0 o 'normale')
#x_normal, y_normal, normal_mask = select_normal_records(x_all, y_all, normal_label=NORMAL_LABEL)
# se volessimo prenderle tutte
x_normal, y_normal = x_all, y_all

print(f"Totale record: {len(x_all)}")
print(f"Record normali trovati: {len(x_normal)}")

results = []
valid_indices = []
invalid_indices = []

for i, ecg in enumerate(x_normal):
    result = check_ecg_quality(ecg, min_valid_leads=9)
    results.append(result)

    if result['global_valid']:
        valid_indices.append(i)
    else:
        invalid_indices.append(i)

        lead_reasons = [
            f"lead {lr['lead_idx']}: {lr['reason']}"
            for lr in result['lead_results']
            if not lr['valid']
        ]

        print(
            f"ECG #{i} NON valido | "
            f"lead invalide: {result['invalid_lead_indices']} | "
            f"motivi: {' ; '.join(lead_reasons)} | "
            f"etichetta: {MAPPING_BY_ID[y_normal[i]]}"
        )

x_normal_valid = x_normal[valid_indices]
x_normal_invalid = x_normal[invalid_indices]

print(f"Validi: {len(x_normal_valid)}")
print(f"Non validi: {len(x_normal_invalid)}")

analizzando val_dataset.h5
Totale record: 7888
Record normali trovati: 7888
ECG #11 NON valido | lead invalide: [2, 4, 5, 6, 10, 11] | motivi: lead 2: high_skewness ; lead 4: high_skewness ; lead 5: high_skewness ; lead 6: high_skewness ; lead 10: high_skewness ; lead 11: high_skewness | etichetta: LA-LL
ECG #141 NON valido | lead invalide: [4, 6, 10, 11] | motivi: lead 4: high_skewness ; lead 6: high_skewness ; lead 10: high_skewness ; lead 11: high_skewness | etichetta: RA-LL
ECG #182 NON valido | lead invalide: [2, 5, 6, 7, 8] | motivi: lead 2: high_skewness ; lead 5: high_skewness ; lead 6: high_skewness ; lead 7: high_skewness ; lead 8: high_skewness | etichetta: LA-LL
ECG #292 NON valido | lead invalide: [3, 6, 10, 11] | motivi: lead 3: high_skewness ; lead 6: high_skewness ; lead 10: high_skewness ; lead 11: high_skewness | etichetta: RL-LA
ECG #319 NON valido | lead invalide: [2, 4, 5, 6, 10, 11] | motivi: lead 2: high_skewness ; lead 4: high_skewness ; lead 5: high_skewness ; 

In [30]:
import os
# creo il dataset togliendo quelli non validi

# Supponiamo di avere già:
# x_normal_valid  -> shape (N_valid, 12, 250) o (N_valid, 250, 12)
# y_normal_valid  -> shape (N_valid,)

orig_name = dataset
base, ext = os.path.splitext(orig_name)
clean_name = base + "_clean" + ext      # -> "test_dataset_clean.h5"

print(f"Salvo file pulito in: {clean_name}")

y_normal_valid = y_normal[valid_indices]
# 1) Costruisci la maschera che esclude 6 e 7
mask_no_6_7 = (y_normal_valid != 6) & (y_normal_valid != 7)

x_clean = x_normal_valid[mask_no_6_7]
y_clean = y_normal_valid[mask_no_6_7]

print(f"Prima: {len(y_normal_valid)}  |  Dopo filtro 6/7: {len(y_clean)}")

with h5py.File(clean_name, "w") as f:
    f.create_dataset("X", data=x_clean)
    f.create_dataset("Y", data=y_clean)

print("Salvataggio completato.")

Salvo file pulito in: val_dataset_clean.h5
Prima: 7772  |  Dopo filtro 6/7: 5836
Salvataggio completato.


In [31]:


for fname in ["train_dataset_clean.h5", "val_dataset_clean.h5", "test_dataset_clean.h5"]:
    with h5py.File(fname, "r") as f:
        y = np.array(f["Y"])
        print(f"\nFile: {fname}")
        print("  Classi uniche Y:", np.unique(y))
        print("  Min:", y.min(), " Max:", y.max())


File: train_dataset_clean.h5
  Classi uniche Y: [0 1 2 3 4 5]
  Min: 0  Max: 5

File: val_dataset_clean.h5
  Classi uniche Y: [0 1 2 3 4 5]
  Min: 0  Max: 5

File: test_dataset_clean.h5
  Classi uniche Y: [0 1 2 3 4 5]
  Min: 0  Max: 5
